In [1]:
%load_ext autoreload
%autoreload 2
%autosave 30

Autosaving every 30 seconds


# Evaluate the raw ensemble and develop a first baseline

In this notebook the train, valid and test datasets are scored using the crps, energy score and variogram score.\
This will be done per lead time.

In [2]:
import os
from itertools import batched
from pathlib import Path

import dask as da  # noqa: F401
import hydra
import numpy as np
import pandas as pd
import torch
import xarray as xr
from einops import rearrange
from tqdm import tqdm

import wandb
from genpp.data import (
    BASE_DIR,
    FC_VARS,
    FORECAST_ENS_FLAT_AGG_PATH,
    FORECAST_ENS_PATH,
    OBSERVATIONS_FLAT_PATH,
)
from genpp.eval.utils import compute_scores_per_leadtime, save_scores_df
from genpp.models.loss import EnergyScore, EnsembleCRPS, VariogramScore

/home/feik/GenPP/.pixi/envs/nb-gpu/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/feik/GenPP/.pixi/envs/nb-gpu/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened bec

In [3]:
config_path = BASE_DIR / "configs" / "data" / "splits"
notebook_dir = Path.cwd()
relative_path = os.path.relpath(config_path, notebook_dir)

with hydra.initialize(version_base=None, config_path=str(relative_path)):
    cfg = hydra.compose(config_name="standard")
cfg = hydra.utils.instantiate(cfg)

In [4]:
def load_data(time_slice: slice) -> tuple[xr.DataArray, xr.DataArray]:
    """Load data for evaluation."""
    ens = xr.open_dataset(
        FORECAST_ENS_PATH,
        chunks={
            "time": "auto",
            "number": -1,
            "latitude": -1,
            "longitude": -1,
            "level": -1,
        },
    )
    ens = ens[FC_VARS]

    # This already contains the cleaned time data
    ens_flat = xr.open_dataset(FORECAST_ENS_FLAT_AGG_PATH)

    ens = (
        ens.sel(time=ens_flat.time)
        .to_dataarray("feature")
        .stack(prediction=("time", "prediction_timedelta"))
        .transpose("prediction", "number", "feature", "longitude", "latitude")
    )

    # Select all predictions for which we have observations
    obs = (
        xr.open_dataset(OBSERVATIONS_FLAT_PATH)
        .sel(time=time_slice)
        .to_dataarray("feature")
        .sel(feature=FC_VARS)  # get them in the correct order
        .transpose("time", "feature", "longitude", "latitude")
    )
    # All times for which we have observations
    obs_times = obs.time.values

    # Mask to only keep prediction times that are in obs_times
    prediction_times = ens.time + ens.prediction_timedelta
    mask = np.isin(prediction_times, obs_times)
    prediction_times = prediction_times[mask]

    # Select only those times in obs
    ens = ens.sel(prediction=mask)
    obs = obs.sel(time=prediction_times.values).rename({"time": "prediction_time"})

    # Do some renaming and sanity checks
    val_prediction = pd.MultiIndex.from_arrays(
        [ens.time.values, ens.prediction_timedelta.values], names=["time", "prediction_timedelta"]
    )
    obs = obs.assign_coords(prediction=("prediction_time", val_prediction)).swap_dims(
        {"prediction_time": "prediction"}
    )

    assert np.all((ens.time + ens.prediction_timedelta).values == obs.prediction_time.values)

    obs = obs.drop_vars("prediction_time")
    return ens, obs

## Compute the Scores

In [5]:
def score(ens: xr.DataArray, obs: xr.DataArray, skip_variogram: bool):
    """Compute scores given ensemble and observations.
    Args:
        ens (xr.DataArray): Ensemble predictions with dimensions (time, number, feature, longitude, latitude)
        obs (xr.DataArray): Observation data with dimensions (time, feature, longitude, latitude)
        skip_variogram (bool): Whether to skip variogram score computation, due to long computation times

    """
    ens_tensor = torch.from_numpy(ens.to_numpy())
    obs_tensor = torch.from_numpy(obs.to_numpy())

    # Compute the scores
    crps_ens = EnsembleCRPS()
    es = EnergyScore(clamp=False)
    vs = VariogramScore(p=0.5)

    # Do all comuptations batched as we might run out of memory otherwise
    batch_indices = batched(range(ens_tensor.shape[0]), 1000)

    crps_per_margin_list = []
    energy_score_per_var_u_list = []
    energy_score_full_u_list = []

    for idx in tqdm(batch_indices, total=len(ens_tensor) // 1000 + 1):
        idx = torch.tensor(idx)
        ens_b = ens_tensor[idx]
        obs_b = obs_tensor[idx]

        crps_per_margin_list.append(crps_ens(ens_b, obs_b))
        # Rearrange so that we compute the scores seperatly for each variable
        x_spatial = rearrange(ens_b, "t n d lat lon -> t d n (lat lon)")
        y_spatial = rearrange(obs_b, "t d lat lon -> t d (lat lon)")

        # _u is for un-reduced scores
        energy_score_per_var_u_list.append(es(x_spatial, y_spatial))

        # Rearrange to compute scores for both variables together
        x_full = rearrange(ens_b, "t n d lat lon -> t n (d lat lon)")
        y_full = rearrange(obs_b, "t d lat lon -> t (d lat lon)")

        energy_score_full_u_list.append(es(x_full, y_full))

    # For the VS there are huge intermediary results, thats why it is computed batchwise in even smaller batches
    if not skip_variogram:
        batch_indices = batched(range(ens_tensor.shape[0]), 10)
        vss_per_var_list = []
        vss_full_u_list = []

        for idx in tqdm(batch_indices, total=len(ens_tensor) // 10 + 1):
            idx = torch.tensor(idx)
            ens_b = ens_tensor[idx]
            obs_b = obs_tensor[idx]
            x_spatial = rearrange(ens_b, "t n d lat lon -> t d n (lat lon)")
            y_spatial = rearrange(obs_b, "t d lat lon -> t d (lat lon)")
            vss_per_var_list.append(vs(x_spatial, y_spatial))
            x_full = rearrange(ens_b, "t n d lat lon -> t n (d lat lon)")
            y_full = rearrange(obs_b, "t d lat lon -> t (d lat lon)")
            vss_full_u_list.append(vs(x_full, y_full))
        variogram_score_per_var_u = torch.cat(vss_per_var_list)
        variogram_score_full_u = torch.cat(vss_full_u_list)

    crps_per_margin = torch.cat(crps_per_margin_list)
    energy_score_per_var_u = torch.cat(energy_score_per_var_u_list)
    energy_score_full_u = torch.cat(energy_score_full_u_list)

    scores = compute_scores_per_leadtime(
        prediction_timedeltas=ens.prediction_timedelta.values,
        crpss=crps_per_margin,
        ess_per_var=energy_score_per_var_u,
        ess_complete=energy_score_full_u,
        vss_per_var=variogram_score_per_var_u if not skip_variogram else None,  # type: ignore
        vss_complete=variogram_score_full_u if not skip_variogram else None,  # type: ignore
        method=None,
    )
    return scores

## Put everything together

In [6]:
scores = {}
for split, time_slice in cfg.items():
    print(f"Scoring split: {split}: {time_slice}")
    ens, obs = load_data(time_slice)
    scores[split] = score(ens, obs, skip_variogram=False)

Scoring split: train: slice('2018-01-03', '2020-12-31', None)


100%|██████████| 1093/1093 [1:47:56<00:00,  5.93s/it]


Processing leadtime 24h with 2187 samples
Processing leadtime 48h with 2187 samples
Processing leadtime 72h with 2185 samples
Processing leadtime 96h with 2183 samples
Processing leadtime 120h with 2181 samples
Scoring split: val: slice('2021-01-01', '2021-12-31', None)


100%|█████████▉| 365/366 [36:09<00:05,  5.94s/it]


Processing leadtime 24h with 730 samples
Processing leadtime 48h with 730 samples
Processing leadtime 72h with 730 samples
Processing leadtime 96h with 730 samples
Processing leadtime 120h with 730 samples
Scoring split: test: slice('2022-01-01', '2022-12-31', None)


100%|█████████▉| 365/366 [36:17<00:05,  5.97s/it]

Processing leadtime 24h with 730 samples
Processing leadtime 48h with 730 samples
Processing leadtime 72h with 730 samples
Processing leadtime 96h with 730 samples
Processing leadtime 120h with 730 samples


## Create a dummy wandb run and save the scores

In [7]:
run = wandb.init(
    project="genpp",
    name="ifs-raw-ensemble-baseline-scores",
    tags=["baseline", "best"],
    config={
        "method": "raw_ensemble",
        "description": "Baseline scores for raw IFS ensemble forecasts",
    },
    dir=BASE_DIR.parent.parent / "outputs" / "BASELINE",
)

wandb: Currently logged in as: feik to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
# ID of the baseline run
run_id = "feik/genpp/utgo5npv"

# Create a wandb run to save the scores
api = wandb.Api()
run = api.run(run_id)
short_run_id = run_id.split("/")[-1]
try:
    # If the run already exists, resume it
    with wandb.init(
        entity=run.entity, project=run.project, id=short_run_id, resume="must"
    ) as active_run:
        active_run.summary.update(scores)

except Exception:
    # If it does not exist, create a new one
    run = wandb.init(
        project="genpp",
        name="ifs-raw-ensemble-baseline-scores",
        tags=["baseline", "best"],
        config={
            "name": "raw_ensemble",
            "description": "Baseline scores for raw IFS ensemble forecasts",
        },
        dir=BASE_DIR.parent.parent / "outputs" / "BASELINE",
        resume="must",
    )

    run.summary.update(scores)

# Also log as a table
records = []
for dataset, metrics in scores.items():
    for metric_name, horizons in metrics.items():
        for horizon, value in horizons.items():
            records.append(("raw_ensemble", dataset, metric_name, horizon, value))
df = pd.DataFrame(records, columns=["method", "dataset", "metric", "horizon", "value"])

save_scores_df(df=df, run_path="/".join(run.path))

try:
    run.finish()
except AttributeError:
    pass

In [10]:
df

,method,dataset,metric,horizon,value
0,raw_ensemble,train,CRPS_combined,24h,0.485566
1,raw_ensemble,train,CRPS_combined,48h,0.553411
2,raw_ensemble,train,CRPS_combined,72h,0.623924
3,raw_ensemble,train,CRPS_combined,96h,0.721872
4,raw_ensemble,train,CRPS_combined,120h,0.839444
...,...,...,...,...,...
130,raw_ensemble,test,VariogramScore_10m_windspeed,24h,174547.937500
131,raw_ensemble,test,VariogramScore_10m_windspeed,48h,197470.953125
132,raw_ensemble,test,VariogramScore_10m_windspeed,72h,222212.859375
133,raw_ensemble,test,VariogramScore_10m_windspeed,96h,251622.703125
